## Load preprocessed hospital data

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("hospital_data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Run get_data.ipynb to create hospital_data.csv first.")

all_data = pd.read_csv(DATA_PATH, dtype={
    "Rndrng_Prvdr_CCN": "string", "Rndrng_Prvdr_Zip5": "string",
    "DRG_Cd": "string", "Rndrng_Prvdr_State_FIPS": "string",
    "Rndrng_Prvdr_RUCA": "string", "data_year": "int64",
    "hsa_id": "Int64", "hsa_classification": "Int64", "ai_class": "Int64",
})
print(f"Loaded {len(all_data):,} rows and {all_data['Rndrng_Prvdr_CCN'].nunique():,} distinct hospitals.")
all_data.head()

Loaded 59,837 rows and 507 distinct hospitals.


,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_City,Rndrng_Prvdr_St,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt,Avg_Mdcr_Pymt_Amt,data_year,hsa_id,hsa_classification,ai_class
0,010022,Floyd Cherokee Medical Center,Centre,400 Northwood Dr,01,35960,AL,3,Metropolitan area low commuting: primary flow ...,897,"ALCOHOL, DRUG ABUSE OR DEPENDENCE WITHOUT REHA...",57,13976.947368,8511.982456,7404.649123,2024,1018,1,1
1,010040,Gadsden Regional Medical Center,Gadsden,1007 Goodyear Avenue,01,35903,AL,1,Metropolitan area core: primary flow within an...,064,INTRACRANIAL HEMORRHAGE OR CEREBRAL INFARCTION...,24,180646.666670,12030.916667,10633.916667,2024,1039,1,1
2,010040,Gadsden Regional Medical Center,Gadsden,1007 Goodyear Avenue,01,35903,AL,1,Metropolitan area core: primary flow within an...,065,INTRACRANIAL HEMORRHAGE OR CEREBRAL INFARCTION...,26,138117.653850,8523.115385,5145.730769,2024,1039,1,1
3,010040,Gadsden Regional Medical Center,Gadsden,1007 Goodyear Avenue,01,35903,AL,1,Metropolitan area core: primary flow within an...,069,TRANSIENT ISCHEMIA WITHOUT THROMBOLYTIC,12,105193.083330,6012.500000,4713.833333,2024,1039,1,1
4,010040,Gadsden Regional Medical Center,Gadsden,1007 Goodyear Avenue,01,35903,AL,1,Metropolitan area core: primary flow within an...,101,SEIZURES WITHOUT MCC,13,129347.692310,10746.000000,5347.230769,2024,1039,1,1


## Category counts
Distinct hospital counts use CCNs across all years and DRGs. A hospital with records in both categories counts once in each category.

In [2]:
category_summary = (
    all_data.groupby("ai_class")
    .agg(rows=("Rndrng_Prvdr_CCN", "size"),
         distinct_hospitals=("Rndrng_Prvdr_CCN", "nunique"),
         distinct_hsas=("hsa_id", "nunique"))
    .reindex([1, 0], fill_value=0)
)
category_summary.insert(0, "category", ["hotspot", "coldspot"])
print(category_summary.to_string())

          category   rows  distinct_hospitals  distinct_hsas
ai_class                                                    
1          hotspot  55578                 449            242
0         coldspot   4259                  58             43
